# Notebook 3 | Conditional Probability

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## Conditioning as slicing and renormalizing

Conditioning restricts the joint $p(B = b, H = h, C = c)$ to the observed event and renormalizes.
For generic random variables $X$ and $Y$, in full form first and then in shorthand,

$$p(X = x \mid Y = y) = \frac{p(X = x, Y = y)}{p(Y = y)}, \qquad p(x \mid y) = \frac{p(x, y)}{p(y)}.$$

## Derivation

1. Slice the joint to the conditioning event, e.g. all states with $B = \text{Porsche}$.
2. The slice sums to the probability of the event, here $p(\text{Porsche}) = 0.3$.
3. Divide the slice by that sum so the conditional distribution is normalized.
4. Read off entries, e.g. $p(\text{black} \mid \text{Porsche}) = 0.12 / 0.3 = 0.4$.

## Worked example (by hand)

Forward direction, restricting to Porsche:

$$p(\text{black} \mid \text{Porsche}) = \frac{0.3 \cdot 0.4}{0.3} = 0.4.$$

Reverse direction, restricting to black cars (a preview of Bayes' theorem):

$$p(\text{Porsche} \mid \text{black}) = \frac{0.12}{0.31} \approx 0.387.$$

In [2]:
import math

import pandas as pd

joint = create_joint_car_distribution()
idx = pd.IndexSlice

# slice to the brand and renormalize: p(h, c | Porsche) = p(Porsche, h, c) / p(Porsche)
conditional = joint.xs("Porsche", level="brand")
conditional = conditional / conditional.sum()
assert math.isclose(conditional.sum(), 1.0)
assert math.isclose(conditional.xs("black", level="color").sum(), (0.3 * 0.4) / 0.3)
assert math.isclose(conditional.xs("black", level="color").sum(), 0.4)

# reverse direction: p(Porsche | black) = p(Porsche, black) / p(black)
p_porsche_black = joint.loc[idx["Porsche", :, "black"],].sum()
p_black = joint.xs("black", level="color").sum()
assert math.isclose(p_black, 0.31)
assert math.isclose(p_porsche_black / p_black, 0.12 / 0.31)
assert math.isclose(p_porsche_black / p_black, 0.387, rel_tol=1e-3)
p_porsche_black / p_black

np.float64(0.3870967741935484)

## Generalization

The same slice-and-renormalize one-liner conditions on any event: a single value, a set of states,
or a value of a derived quantity such as high horsepower. Conditioning on the full joint always returns
a normalized distribution over the remaining variables.

In [3]:
# conditioning on high horsepower: p(brand | horsepower >= 500)
high_hp = joint[joint.index.get_level_values("horsepower") >= 500]
posterior = high_hp.groupby(level="brand").sum()
posterior = posterior / posterior.sum()
assert math.isclose(posterior.sum(), 1.0)
posterior

brand
Ferrari    0.7
Porsche    0.3
Name: p, dtype: float64

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Probability", https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.